# Infidelity

La classe CircuitShape deve memorizzare (opzionalmente) affiancata ad ogni timestep la sua infidelity. Si aggiorna il suo costruttore in modo che le si possano passare come parametri opzionali il vettore delle infidelity dei singoli gate e la matrice dei crosstalks

In [45]:
import scipy.sparse as sp
import numpy as np

In [46]:
class CircuitShape:
    def __init__(self,n_qubits,gates,eq_dir="equivalenze v2",infidelities=None, crosstalks=None):
        self._eq_dir = eq_dir
        self._from_gates = gates
        self._n_qubits = n_qubits
        self._steps = []
        if infidelities is None:
            self._infidelities = None
            self._find_steps(gates,[],0)
        else:
            self._infidelities = []
            if crosstalks is None:
                n_gates = sum(len(gates[key]) for key in gates.keys())
                crosstalks = np.zeros((n_gates,n_gates),dtype=np.float64)
            self._find_steps_with_infidelities(gates,[],0,[],infidelities,crosstalks)
        
    
    def _find_steps_with_infidelities(self,gates,step,length,included_gates,infidelities,crosstalks):
        if length == self._n_qubits:
            self._steps.append(step)
            infidelity = 0
            for i in range(len(included_gates)):
                infidelity += infidelities[included_gates[i]]
                for j in range(i+1,len(included_gates)):
                    infidelity += crosstalks[included_gates[i]][included_gates[j]]
            self._infidelities.append(infidelity)
        else:
            for gate_length in gates.keys():
                start_index = sum(len(gates[key]) for key in range(1,gate_length))
                i=0
                for gate in gates[gate_length]:
                    if length+gate_length <= self._n_qubits:
                        new_step = step.copy()
                        for block in gate:
                            new_step.append(block)
                        self._find_steps_with_infidelities(gates,new_step,length+gate_length,included_gates+[start_index+i],infidelities,crosstalks)
                    i+=1
    
    # recursively finds possible timesteps given a set of gates and a number of qubits  
    def _find_steps(self,gates,step,length):
        if length == self._n_qubits:
            self._steps.append(step)
        else:
            for gate_length in gates.keys():
                for gate in gates[gate_length]:
                    if length+gate_length <= self._n_qubits:
                        new_step = step.copy()
                        for block in gate:
                            new_step.append(block)
                        self._find_steps(gates,new_step,length+gate_length)
    
    # returns quantum state from single step list
    def state_of_step(self,step):
        state = np.zeros((len(self._steps)),dtype=np.float64)
        state[self._steps.index(step)] = 1.
        return state
    # return step from single step quantum state
    def step_of_state(self,state):
        if not state.is_classical(): raise Exception("The state is not classical")
        return self._steps[np.where(state!=0)[0][0]]
    def index_of_state(self,state):
        if not state.is_classical(): raise Exception("The state is not classical")
        return np.where(state!=0)[0][0]
    def get_n_qubits(self):
        return self._n_qubits
    def get_step_index(self,step):
        return self._steps.index(step)
    def get_n_steps(self):
        return len(self._steps)
    def _getID(self):
        return self._steps[0]
    
    def get_steps(self):
        return self._steps.copy()
    
    def print_steps(self,auto_print=True):
        out = ""
        for i in range(len(self._steps)):
            out += f"{i}: {self._steps[i]}" 
            if self._infidelities is not None:
                out += f"\t\t Inf.={self._infidelities[i]}"
            out += "\n"
        if auto_print: print(out)
        return out
    
            
    def find_equivalences(self,d_max):
        redundant_eq = [ [[(i),(i)] for i in range(len(self._steps))] ]
        #final_eq = []

        for d in range(2,d_max+1):
            ids = list(it.product(range(len(self._steps)), repeat=d))
            redundant_eq.append([])
            #final_eq.append([])
            found = []
            
            with open(f"{self._eq_dir}/eq_d={d}.csv", 'w',newline="") as fout:
                writer = csv.writer(fout, delimiter="\t")
                writer.writerow(["n_qubits:" +str(self._n_qubits)])
                writer.writerow(["gates:"])
                for key in self._from_gates.keys():
                    writer.writerow(self._from_gates[key])
                writer.writerow(["equivalences:"])                    
                
                for i in range(len(ids)):
                    print(i,"/",len(ids))
                    redundant_eq[d-1].append([])
                    k=0
                    while k < len(found):
                        if found[k][1] == i:
                            redundant_eq[d-1][i] = [[ids[i],eq[1]] if ids[i]!=eq[1] else [ids[i],eq[0]] for eq in redundant_eq[d-1][found[k][0]]]
                            k = len(found)+1
                        else:
                            k += 1
                    if k == len(found):
                        for j in range(i+1, len(ids)):
                            # check if the rule can be obtained composing shorter eq. rules
                            d_i = 1
                            while d_i < d:
                                if [ids[i][:d_i],ids[j][:d_i]] in redundant_eq[d_i-1] or [ids[i][d-d_i:],ids[j][d-d_i:]] in redundant_eq[d_i-1]:
                                    d_i = d+1
                                else:
                                    d_i += 1
                            if d_i == d and CircuitState.equivalent_from_steps([self._steps[k] for k in ids[i]],[self._steps[k] for k in ids[j]]):
                                #final_eq[d-2].append([ids[i],ids[j]])
                                writer.writerow([step_id for step_id in ids[i]+ids[j]])
                                found.append([i,j])
                                redundant_eq[d-1][i].append([ids[i],ids[j]])
                redundant_eq[d-1] = list(it.chain(*redundant_eq[d-1]))

Si testa il calcolo delle infidelities sui soliti gates

In [47]:
n_qubits = 4
gates = { 
    1 : [ ["ID"] ],                     #0
    2 : [   ["Sc","St"],                #1
            ["CZc","CZt"] ],            #2
    3 : [ ["CZc","ID","CZt"] ],           #3
    4 : [ ["CZc","CZc","CZt","CZt"] ]       #4
}

infidelities = [0,3,1,1.5,5]
crosstalks = [
    [0,0,0,0,0],
    [0,1,1,0,0],
    [0,1,1,0,0],
    [0,0,0,0,0],
    [0,0,0,0,0]
]

shape = CircuitShape(n_qubits,gates,infidelities=infidelities,crosstalks=crosstalks)
shape.print_steps(); 

0: ['ID', 'ID', 'ID', 'ID']		 Inf.=0
1: ['ID', 'ID', 'Sc', 'St']		 Inf.=3
2: ['ID', 'ID', 'CZc', 'CZt']		 Inf.=1
3: ['ID', 'Sc', 'St', 'ID']		 Inf.=3
4: ['ID', 'CZc', 'CZt', 'ID']		 Inf.=1
5: ['ID', 'CZc', 'ID', 'CZt']		 Inf.=1.5
6: ['Sc', 'St', 'ID', 'ID']		 Inf.=3
7: ['Sc', 'St', 'Sc', 'St']		 Inf.=7
8: ['Sc', 'St', 'CZc', 'CZt']		 Inf.=5
9: ['CZc', 'CZt', 'ID', 'ID']		 Inf.=1
10: ['CZc', 'CZt', 'Sc', 'St']		 Inf.=5
11: ['CZc', 'CZt', 'CZc', 'CZt']		 Inf.=3
12: ['CZc', 'ID', 'CZt', 'ID']		 Inf.=1.5
13: ['CZc', 'CZc', 'CZt', 'CZt']		 Inf.=5



A questo punto va calcolata l'H di infidelity, di cui si deve occupare la classe CircuitOperations

In [52]:
class CircuitOperations:
    def __init__(self,shape):  
        self._shape = shape
        self._hops = np.zeros((len(shape._steps),len(shape._steps)),dtype=object)
        for i in range(len(shape._steps)):
            for j in range(len(shape._steps)):
                self._hops[i,j] = sp.csr_matrix( np.kron(
                    np.conj(shape.state_of_step(shape._steps[j])).reshape(-1,1),
                    shape.state_of_step(shape._steps[i])
                ), dtype='complex')

    # moves the single-step-circuit state to "to" single-step-circuit state
    def hop(self,to,circuit):
        if not circuit.get_n_steps() == 1: raise Exception("The circuit is not single-step")
        circuit._state = self._hops[circuit._shape.index_of_state(circuit._state),to].dot(circuit._state) 
        
    def get_H_eps(self,ids1,ids2,n_steps):
        id = sp.identity(len(self._hops), format='csr')
        H_eps = sp.csr_matrix((len(self._hops)**n_steps,len(self._hops)**n_steps), dtype='complex')
        for i_eq in range(0,n_steps+1-len(ids1)):
            H_eps_i = sp.identity(1,format='csr') 
            for i in range(0,i_eq):
                H_eps_i = sp.kron(H_eps_i,id)
            for i in range(0,len(ids1)):
                H_eps_i = sp.kron(H_eps_i,self._hops[ids1[i],ids2[i]])
            for i in range(i_eq+len(ids1),n_steps):
                H_eps_i = sp.kron(H_eps_i,id)
            H_eps += H_eps_i
        
        return H_eps + H_eps.getH()
    
    def get_H(self, dmax, n_steps):
        H = sp.csr_matrix((len(self._hops)**n_steps,len(self._hops)**n_steps), dtype='complex')
        for d in range(2,dmax+1):
            filename = f"{self._shape._eq_dir}/eq_d={d}.csv"
            with open(filename, 'r', newline='') as fin:
                reader = csv.reader(fin, delimiter="\t")
                #skips the header
                row = None
                while(row != "equivalences:"):
                    row = next(reader)[0]
                
                for row in reader:
                    ids1 = [int(id) for id in row[:d]]
                    ids2 = [int(id) for id in row[d:]]
                    H += self.get_H_eps(ids1,ids2,n_steps)
        return H
    
    def get_HI(self, n_steps):
        if self._shape._infidelities is None: raise Exception("infidelities not provided in shape's constructor")
        HI_step = sp.csr_matrix((self._shape.get_n_steps(),self._shape.get_n_steps()), dtype='complex')
        for i in range(self._shape.get_n_steps()):
            HI_step += self._shape._infidelities[i] * self._hops[i,i]
        
        id = sp.identity(self._shape.get_n_steps(), format='csr')
        HI = sp.csr_matrix((self._shape.get_n_steps()**n_steps,self._shape.get_n_steps()**n_steps), dtype='complex')
        for i in range(n_steps):
            HI_i = sp.identity(1,format='csr') 
            for _ in range(0,i):
                HI_i = sp.kron(HI_i,id)
            HI_i = sp.kron(HI_i,HI_step)
            for _ in range(0,n_steps-i-1):
                HI_i = sp.kron(HI_i,id)
            HI += HI_i
        
        return HI
    
    def multiply(operator,circuit_state):
        final_state = operator.dot(circuit_state._state)
        return CircuitState(circuit_state._shape,state=final_state)

    def expected_value_from_state(operator, state):
        return np.vdot(state,operator.dot(state))
    def dev_std_from_state(operator,state):
        return np.sqrt( np.dot(state, operator.dot(operator).dot(state)) - np.vdot(state,operator.dot(state))**2 )
        
    def expected_value(operator,circuit_state):
        return CircuitOperations.expected_value_from_state(operator, circuit_state._state)
    def dev_std(operator, circuit_state):
        return CircuitOperations.dev_std_from_state(operator, circuit_state._state)
    

E si testa su diversi ciruiti

In [53]:
from QuantumCompiling import CircuitState

op = CircuitOperations(shape)
HI = op.get_HI(4)


c1 = CircuitState.from_steps_ids([1,0,7,13],shape)



print("Valore atteso:",CircuitOperations.expected_value(HI,c1))
print("Deviazione standard:",CircuitOperations.dev_std(HI,c1))

print(c1.draw())
c1 = CircuitOperations.multiply(HI,c1)
for i in np.where(c1._state != 0)[0]:
    print("Infidelity:",np.real(c1._state[i]))

Valore atteso: (15+0j)
Deviazione standard: 0j
                 
q_0: ─X─────■────
      │     │    
q_1: ─X─────┼──■─
            │  │ 
q_2: ─X──X──■──┼─
      │  │     │ 
q_3: ─X──X─────■─
                 
Infidelity: 15.0
